In [4]:
%load_ext autoreload
%autoreload 2
import dataclasses
from functools import partial

import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P

from moe.ra2a import make_ra2a_3d
from tests.utils import generate_data

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [124]:
def make_split_ra2a(compute_fn):

  @partial(jax.custom_vjp, nondiff_argnames=("axis_name",))
  def ra2a_split(payloads, args, axis_name: str):
    _start_fn, _wait_fn = make_ra2a_3d(axis_name=axis_name)

    futures = []
    for (src, output, input_offsets, send_sizes, output_offsets, recv_sizes) in payloads:
      future = _start_fn(src, output, input_offsets, send_sizes, output_offsets, recv_sizes)
      futures.append(future)

    outs = []
    for future in futures:
      out = _wait_fn(future)
      outs.append(out)

    y = compute_fn(*args)

    # outs = [
    #   jax.lax.ragged_all_to_all(src, output, input_offsets, send_sizes, output_offsets, recv_sizes,
    #                             axis_name=axis_name)
    #   for (src, output, input_offsets, send_sizes, output_offsets, recv_sizes) in payloads
    # ]

    return outs, y

  def ra2a_split_fwd(payloads, args, axis_name: str):
    res = ([[payload[0].shape] + list(payload[2:]) for payload in payloads], args)
    return ra2a_split(payloads, args, axis_name), res

  def ra2a_split_bwd(axis_name: str, res, g):
    _start_fn, _wait_fn = make_ra2a_3d(axis_name=axis_name)
    payloads, args = res
    tangents = g[0]

    futures = []
    for tangent, payload in zip(tangents, payloads, strict=True):
      (src_shape, input_offsets, send_sizes, output_offsets, recv_sizes) = payload
      inv_send_sizes, inv_recv_sizes = recv_sizes, send_sizes
      inv_input_offsets = jax.lax.all_to_all(output_offsets, axis_name, split_axis=0, concat_axis=0)
      inv_output_offsets = jax.lax.all_to_all(input_offsets, axis_name, split_axis=0, concat_axis=0)
      buf = jax.lax.empty(src_shape, dtype=tangent.dtype)
      future = _start_fn(tangent, buf, inv_input_offsets, inv_send_sizes, inv_output_offsets, inv_recv_sizes)
      futures.append(future)

    douts = []
    for future in futures:
      dout = _wait_fn(future)
      douts.append(tuple([dout] + [None] * 5))

    dcompute = jax.vjp(compute_fn, *args)[1](g[1])

    #douts = []
    #for (tangent, (src_shape, input_offsets, send_sizes, output_offsets, recv_sizes)) in zip(g[0], payloads, strict=True):
    #  inv_send_sizes, inv_recv_sizes = recv_sizes, send_sizes
    #  inv_input_offsets = jax.lax.all_to_all(output_offsets, axis_name, split_axis=0, concat_axis=0)
    #  inv_output_offsets = jax.lax.all_to_all(input_offsets, axis_name, split_axis=0, concat_axis=0)
    #  buf = jax.lax.empty(src_shape, tangent.dtype)
    #  dout = jax.lax.ragged_all_to_all(tangent, buf, inv_input_offsets, inv_send_sizes, inv_output_offsets, inv_recv_sizes, axis_name=axis_name)
    #  douts.append(tuple([dout]  + [None] * 5))

    return douts, dcompute

  ra2a_split.defvjp(ra2a_split_fwd, ra2a_split_bwd)

  return ra2a_split

In [125]:
axis_name = "x"
devices = jax.devices()
mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit)
jax.sharding.set_mesh(mesh)

In [126]:
x, meta = generate_data(1024, 1024, device_num=len(devices), axis_name=axis_name)
x = x.reshape((x.shape[0], 8, -1))

In [127]:
def compute_fn(x):
  return jnp.zeros_like(x)


ra2a_split = make_split_ra2a(compute_fn)

In [128]:
@jax.jit
def fn_ref(x, meta):
  @partial(jax.shard_map, out_specs=P(axis_name), check_vma=False)
  def inner(x, meta):
    buffer = jax.lax.empty((2 * x.shape[0], *x.shape[1:]), x.dtype)
    payload = (x, buffer, *dataclasses.astuple(meta))
    x_ra2a = [jax.lax.ragged_all_to_all(*payload, axis_name=axis_name)]
    #x_ra2a, y = ra2a_split([payload], (x,), axis_name=axis_name)
    y = compute_fn(x)
    return x_ra2a, y

  return inner(x, meta)

In [129]:
@jax.jit
def fn(x, meta):
  @partial(jax.shard_map, out_specs=P(axis_name), check_vma=False)
  def inner(x, meta):
    buffer = jax.lax.empty((2 * x.shape[0], *x.shape[1:]), x.dtype)
    payload = (x, buffer, *dataclasses.astuple(meta))
    x_ra2a, y = ra2a_split([payload], (x,), axis_name=axis_name)
    return x_ra2a, y

  return inner(x, meta)

In [130]:
o, vjp_fn = jax.vjp(fn, x, meta)
o_ref, vjp_ref_fn = jax.vjp(fn_ref, x, meta)

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


In [131]:
jax.tree.structure(o_ref)

PyTreeDef(([*], *))

In [132]:
jax.tree.structure(o)

PyTreeDef(([*], *))

In [133]:
r = jax.tree.map(lambda x: jax.random.normal(jax.random.key(0), x.shape, x.dtype, out_sharding=x.sharding), o)

In [134]:
do = vjp_fn(r)
do_ref = vjp_ref_fn(r)
print(jnp.sum(jnp.abs(do_ref[0] - do[0])))
# print(do)
# print("--------------------------------------------------------------")

ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?
ERROR:root:AsyncCopyDescriptor was not used. Did you mean to call `start` or `wait` on it?


0
